# 🚀 Entrenamiento Qwen 2.5 - Acervo UABC (Derecho, Administración, Salud y Humanidades)
### Proyecto MTAO - FCITEC UABC | Autor: L.S.C. Ana Gabriela Molina Moreno / Desarrollador: Adrián

Este notebook entrena el modelo **Qwen 2.5** con las **3 partes restantes del catálogo** (7,500 ejemplos reales de la Biblioteca Central Mexicali) **de a chingadazo** en una sola sesión de GPU T4 de Google Colab.

**Dominios incluidos:**
- ⚖️ **Parte 2:** Derecho, Amparo, Ciencias Sociales, Historia de México/BC, Humanidades y Psicología.
- 📊 **Parte 3:** Administración, Contabilidad, Finanzas, Mercadotecnia y Economía.
- 🩺 **Parte 4:** Medicina, Fisiología, Farmacología, Enfermería, Ciencias Básicas (Física/Química) y Artes.

## Paso 1: Instalar dependencias optimizadas (Unsloth)

In [ ]:
# Instalación de Unsloth y aceleradores
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install unsloth_zoo
!pip install --no-deps trl peft accelerate bitsandbytes datasets

## Paso 2: Cargar el Modelo Base Qwen 2.5 en 4-bit

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
dtype = None # Auto detección (Float16 en T4)
load_in_4bit = True # Cuantización en 4 bits para mínimo consumo de VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

## Paso 3: Configurar Adaptadores LoRA Especializados (r=32, alpha=64)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## Paso 4: Cargar los 7,500 ejemplos del Dataset
> Sube el archivo `dataset_koha_3partes_completas.jsonl` a la pestaña de archivos (📁) de Colab.
> *(O si prefieres, sube los 3 archivos individuales: parte 2, parte 3 y parte 4, la celda detecta ambos automáticamente)*.

In [ ]:
import os
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

def formatting_prompts_func(examples):
    convs = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convs]
    return { "text" : texts }

# Detecta si subiste el archivo combinado o los 3 por separado
if os.path.exists("dataset_koha_3partes_completas.jsonl"):
    data_files = "dataset_koha_3partes_completas.jsonl"
    print(f"Cargando archivo único: {data_files}")
else:
    data_files = [
        "dataset_koha_parte2_ciencias_sociales_derecho.jsonl",
        "dataset_koha_parte3_administracion_economia.jsonl",
        "dataset_koha_parte4_salud_ciencias_basicas.jsonl"
    ]
    print(f"Cargando los 3 archivos individuales: {data_files}")

dataset = load_dataset("json", data_files=data_files, split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"¡Dataset de 3 partes cargado! Muestras totales: {len(dataset)}")


## Paso 5: Entrenar el Modelo con las 3 Partes (~18-20 min en GPU T4)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Con 7,500 ejemplos y batch efectivo de 16, 900 pasos = ~2 épocas completas
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4, # Batch efectivo = 16
        warmup_steps = 30,
        max_steps = 900, # ~2 épocas completas sobre 7,500 ejemplos (~18 a 20 min en T4)
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 20,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("Iniciando entrenamiento masivo (900 pasos / 7,500 ejemplos)...\n")
trainer_stats = trainer.train()
print("\n¡Entrenamiento completado exitosamente!")


## Paso 6: Prueba en Vivo con Consultas Multidisciplinarias

In [ ]:
FastLanguageModel.for_inference(model)

preguntas_prueba = [
    "¿Qué onda pa, ocupo libros de derecho constitucional mexicano y amparo?",
    "paro, recomiéndame libros de contabilidad financiera y costos",
    "¿Tienen disponible en sala el libro de fisiología médica de Guyton?",
    "a qué hora abren los sábados en la biblioteca central?"
]

for p in preguntas_prueba:
    messages = [
        {"role": "system", "content": "Eres el asistente inteligente de la Biblioteca Central Campus Mexicali (UABC). Tu función es analizar la consulta del usuario en lenguaje natural y extraer la intención y parámetros de búsqueda para la API de KOHA en formato JSON."},
        {"role": "user", "content": p}
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize = True, add_generation_prompt = True, return_tensors = "pt").to("cuda")
    outputs = model.generate(input_ids = inputs, max_new_tokens = 100, use_cache = True)
    resp = tokenizer.batch_decode(outputs)[0]
    print(f"Pregunta: {p}")
    print("JSON:", resp.split("<|im_start|>assistant")[-1].replace("<|im_end|>", "").strip())
    print("-"*60)


## Paso 7: Exportar a Formato GGUF y Descargar

In [ ]:
# Exportar a GGUF (q4_k_m)
model.save_pretrained_gguf("qwen_koha_uabc_model", tokenizer, quantization_method = "q4_k_m")
print("¡Modelo GGUF generado en qwen_koha_uabc_model_gguf!")

# Iniciar descarga directa
from google.colab import files
files.download("qwen_koha_uabc_model_gguf/Qwen2.5-0.5B-Instruct.Q4_K_M.gguf")
